## 1.对NEU-DET数据集进行分析

In [1]:
import os
import xml.etree.ElementTree as ET
from collections import Counter
# 数据集路径
dataset_path = r'c:\Users\fengyarui1\Desktop\ReForm‑Net\NEU-DET'
images_path = os.path.join(dataset_path, 'IMAGES')
annotations_path = os.path.join(dataset_path, 'ANNOTATIONS')
# 分析图像文件
image_files = [f for f in os.listdir(images_path) if f.endswith('.jpg')]
print(f"总图像数量: {len(image_files)}")

总图像数量: 1800


In [2]:
# 提取缺陷类型
defect_types = []
for img_file in image_files:
    defect_type = img_file.split('_')[0]
    defect_types.append(defect_type)

# 统计每种缺陷类型的数量
defect_counts = Counter(defect_types)
print("\n缺陷类型分布:")
for defect, count in defect_counts.items():
    print(f"{defect}: {count}")


缺陷类型分布:
crazing: 300
inclusion: 300
patches: 300
pitted: 300
rolled-in: 300
scratches: 300


In [3]:
# 分析标注文件
print("\n标注文件分析:")
annotation_files = [f for f in os.listdir(annotations_path) if f.endswith('.xml')]
print(f"总标注文件数量: {len(annotation_files)}")

# 分析标注信息
bbox_counts = []
for ann_file in annotation_files[:5]:  # 分析前5个标注文件
    ann_path = os.path.join(annotations_path, ann_file)
    tree = ET.parse(ann_path)
    root = tree.getroot()
    objects = root.findall('.//object')
    bbox_counts.append(len(objects))
    print(f"{ann_file}: {len(objects)} 个缺陷")


标注文件分析:
总标注文件数量: 1800
crazing_1.xml: 1 个缺陷
crazing_10.xml: 2 个缺陷
crazing_100.xml: 4 个缺陷
crazing_101.xml: 1 个缺陷
crazing_102.xml: 2 个缺陷


## 2.使用 YOLO11 实现 NEU-DET 数据集的表面缺陷检测
### 2.1环境搭建
安装 Ultralytics 库

In [ ]:
# 安装 Ultralytics 库
!pip install ultralytics -i https://pypi.tuna.tsinghua.edu.cn/simple

### 2.2转换标注格式
YOLO11 使用 YOLO 格式的标注文件（.txt），需要将 XML 转换为 YOLO 格式：

In [ ]:
import os
import xml.etree.ElementTree as ET

def convert_xml_to_yolo(xml_dir, output_dir, class_names):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for xml_file in os.listdir(xml_dir):
        if not xml_file.endswith('.xml'):
            continue
        
        tree = ET.parse(os.path.join(xml_dir, xml_file))
        root = tree.getroot()
        
        img_width = int(root.find('size/width').text)
        img_height = int(root.find('size/height').text)
        
        yolo_lines = []
        for obj in root.findall('object'):
            class_name = obj.find('name').text
            if class_name not in class_names:
                continue
            class_id = class_names.index(class_name)
            
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)
            
            # 转换为 YOLO 格式（归一化坐标）
            x_center = (xmin + xmax) / 2 / img_width
            y_center = (ymin + ymax) / 2 / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height
            
            yolo_lines.append(f"{class_id} {x_center} {y_center} {width} {height}")
        
        # 保存 YOLO 格式标注文件
        txt_file = os.path.join(output_dir, xml_file.replace('.xml', '.txt'))
        with open(txt_file, 'w') as f:
            f.write('\n'.join(yolo_lines))

# 定义缺陷类别（修正后的类名）
class_names = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']

# 转换标注文件
xml_dir = r'c:\Users\fengyarui1\Desktop\ReForm‑Net\NEU-DET\ANNOTATIONS'
output_dir = r'c:\Users\fengyarui1\Desktop\ReForm‑Net\NEU-DET\labels'
convert_xml_to_yolo(xml_dir, output_dir, class_names)

### 2.2创建数据集配置文件
创建 neu-det.yaml 文件，定义数据集路径和类别：


In [ ]:
# 数据集根目录（相对于 yaml 文件的路径）
path: ../NEU-DET  

# 训练/验证/测试图片路径（相对于 path）
train: images/train
val: images/val
test: images/test  

# 类别数和名称（必须和你的标签 class_id 对应）
nc: 6
names: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']

编写代码使得文件路径与上面的配置文件相匹配 训练/验证/测试比例为8：1：1

In [2]:
import os
import shutil
import random

base_path = r'c:\Users\fengyarui1\Desktop\ReForm‑Net\NEU-DET'
images_path = os.path.join(base_path, 'images')
labels_path = os.path.join(base_path, 'labels')

splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(os.path.join(base_path, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(base_path, 'labels', split), exist_ok=True)

image_files = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
image_files = [f for f in image_files if os.path.isfile(os.path.join(images_path, f))]

random.seed(42)
random.shuffle(image_files)

total = len(image_files)
train_end = int(total * 0.8)
val_end = int(total * 0.9)

train_files = image_files[:train_end]
val_files = image_files[train_end:val_end]
test_files = image_files[val_end:]

def move_files(file_list, split):
    for img_file in file_list:
        base_name = os.path.splitext(img_file)[0]
        label_file = base_name + '.txt'
        
        src_img = os.path.join(images_path, img_file)
        dst_img = os.path.join(base_path, 'images', split, img_file)
        
        src_label = os.path.join(labels_path, label_file)
        dst_label = os.path.join(base_path, 'labels', split, label_file)
        
        if os.path.exists(src_img):
            shutil.move(src_img, dst_img)
        if os.path.exists(src_label):
            shutil.move(src_label, dst_label)

move_files(train_files, 'train')
move_files(val_files, 'val')
move_files(test_files, 'test')

print(f"总图片数: {total}")
print(f"训练集: {len(train_files)}")
print(f"验证集: {len(val_files)}")
print(f"测试集: {len(test_files)}")

总图片数: 1800
训练集: 1440
验证集: 180
测试集: 180


### 2.3模型训练
选择 YOLO11 模型
- yolo11n.pt ：轻量级模型，适合边缘设备
- yolo11s.pt ：小型模型，平衡速度和精度
- yolo11m.pt ：中型模型，更高精度

In [2]:
from ultralytics import YOLO
import time
import torch
# 加载模型
model = YOLO('yolo11n.pt')

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='YOLO11-baseline',
    device=0,                  
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # 论文Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)
# ===================== 训练结束后自动输出：FPS、参数量、计算量 =====================
print("\n" + "="*60)
print("            训练完成！模型性能指标计算中...")
print("="*60)

# 1. 输出参数量 & GFLOPs
model_info = model.info()  # 会自动打印 params, GFLOPs

# 2. 测试 FPS
dummy_input = torch.randn(1, 3, 200, 200).to("cuda:0")  # 输入尺寸 200×200

# 预热
for _ in range(10):
    model(dummy_input)

# 测速
start_time = time.time()
for _ in range(100):
    model(dummy_input)
torch.cuda.synchronize()
infer_time = (time.time() - start_time) / 100
fps = 1.0 / infer_time

# 最终输出
print("\n" + "="*60)
print(f"模型输入尺寸: 200 × 200")
print(f"单张图片推理时间: {infer_time:.4f} 秒")
print(f"模型 FPS: {fps:.2f}")
print("="*60)

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11-baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=50,

ValueError: torch.Tensor inputs should be BCHW i.e. shape(1, 3, 640, 640) divisible by stride 32. Input shape(1, 3, 200, 200) is incompatible.

### 2.4模型推理

In [2]:
from ultralytics import YOLO
# 加载最佳模型
model = YOLO('runs/detect/YOLO11-baseline/weights/best.pt')

In [11]:
# 对单张图像进行推理
results = model.predict(
    'NEU-DET/images/train/crazing_1.jpg',  # 图像路径
    save=True,                      # 保存结果
    conf=0.1,                       # 置信度阈值
    iou=0.5,                        # IoU 阈值
    imgsz=200                        # 输入图像尺寸
)
# 对整个文件夹进行推理
results = model.predict(
    'NEU-DET/images/test/',
    save=True,
    conf=0.1,
    iou=0.5,
    imgsz=200
)


WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
image 1/1 /root/NEU-DET/images/train/crazing_1.jpg: 224x224 5 crazings, 6.6ms
Speed: 0.4ms preprocess, 6.6ms inference, 0.8ms postprocess per image at shape (1, 3, 224, 224)
Results saved to /root/runs/detect/predict

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
image 1/180 /root/NEU-DET/images/test/crazing_110.jpg: 224x224 7 crazings, 6.5ms
image 2/180 /root/NEU-DET/images/test/crazing_12.jpg: 224x224 4 crazings, 6.4ms
image 3/180 /root/NEU-DET/images/test/crazing_145.jpg: 224x224 3 crazings, 6.4ms
image 4/180 /root/NEU-DET/images/test/crazing_148.jpg: 224x224 5 crazings, 6.3ms
image 5/180 /root/NEU-DET/images/test/crazing_154.jpg: 224x224 4 crazings, 6.5ms
image 6/180 /root/NEU-DET/images/test/crazing_158.jpg: 224x224 3 crazings, 6.6ms
image 7/180 /root/NEU-DET/images/test/crazing_179.jpg: 224x224 5 crazings, 11.0ms
image 8/180 /root/NEU-DET/images/test/crazing_183.jpg: 224x224

### 2.5模型评估

In [15]:
model = YOLO('runs/detect/YOLO11-baseline/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 927.6±269.8 MB/s, size: 8.9 KB)
val: Scanning /root/NEU-DET/labels/val... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 2.6Kit/s 0.1s
val: New cache created: /root/NEU-DET/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 18.0it/s 0.7s0.1s
                   all        180        396       0.67      0.714      0.746      0.437
               crazing         32         66      0.404      0.227      0.295      0.112
             inclusion         38        106      0.685      0.774      0.762      0.419
               patches         30         69      0.785      0.957      0.945      0.607


## 3.使用ReForm-Net 实现 NEU-DET 数据集的表面缺陷检测
### 3.1第一步：RGA 召回引导标注（核心数据实现）
RGA 是**标签重建引擎**，解决矩形框包含大量背景、不规则缺陷标注不准的问题。

#### 1. 实现流程
##### （1）阶段1：瓶颈类别诊断
1. 用**原始标签**训练标准 YOLO11 基线。
2. 在验证集计算**每类缺陷的召回率 Recall**
3. 设定阈值 **θ=0.5**：在2.5小节当中，实验得Recall<0.5的有crazing、pitted_serface、rolled-in_scale、scratches。

##### （2）阶段2：网格标签重构
对瓶颈类别的**原始矩形框直接丢弃**，改用**网格分块重建标签**：
1. 在原框内生成 **N×N 网格**（实验确定 **2×2 最优**）。
2. 逐格判断：**格子内包含缺陷 → 标记为正样本（patch label）**。
3. 任务转换：从“矩形框回归” → **网格块二分类**。

**计算过程**
1. 原始标签含义：类别ID：0   ,x中心：0.4875     ,y中心：0.49      ,宽度w：0.955     ,高度h：0.96
2. 2*2网格重构：一个大框分为4个小框
* dw = w / 2.0
* dh = h / 2.0
* 4 个小框中心坐标:
* 左上：x - dw/2，y - dh/2
* 右上：x + dw/2，y - dh/2
* 左下：x - dw/2，y + dh/2
* 右下：x + dw/2，y + dh/2
3. 代入计算的4 个小框中心坐标（保留6位小数）:
* dw = 0.955 / 2 = 0.4775
* dh = 0.96 / 2 = 0.48
* 左上：0 0.248750 0.250000 0.47750000 0.480000
* 右上：0 0.726250 0.250000 0.47750000 0.480000
* 左下：0 0.248750 0.730000 0.47750000 0.480000
* 右下：0 0.726250 0.730000 0.47750000 0.480000


In [ ]:
import os
import numpy as np

# ===================== 配置 =====================
BOTTLENECK_CLASSES = {0}  # 瓶颈类别

# 原始标签路径
INPUT_TRAIN = "./NEU-DET/labels/train"
INPUT_VAL   = "./NEU-DET/labels/val"
INPUT_TEST  = "./NEU-DET/labels/test"

# 重构后输出路径
OUTPUT_TRAIN = "./NEU-DET/labels/train_rga"
OUTPUT_VAL   = "./NEU-DET/labels/val_rga"
OUTPUT_TEST  = "./NEU-DET/labels/test_rga"

# 自动创建文件夹
os.makedirs(OUTPUT_TRAIN, exist_ok=True)
os.makedirs(OUTPUT_VAL, exist_ok=True)
os.makedirs(OUTPUT_TEST, exist_ok=True)

# ===================== RGA 2×2 网格重构函数 =====================
def rga_reform_2x2(txt_path, save_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        cls, x, y, w, h = map(float, line.split())
        cls = int(cls)

        # 非瓶颈类保持不变
        if cls not in BOTTLENECK_CLASSES:
            new_lines.append(line)
            continue

        # 瓶颈类 → 2×2 切分
        dw = w / 2.0
        dh = h / 2.0
        grids = [
            (x - dw/2, y - dh/2, dw, dh),
            (x + dw/2, y - dh/2, dw, dh),
            (x - dw/2, y + dh/2, dw, dh),
            (x + dw/2, y + dh/2, dw, dh),
        ]
        for gx, gy, gw, gh in grids:
            new_lines.append(f"{cls} {gx:.6f} {gy:.6f} {gw:.6f} {gh:.6f}")

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_lines))

# ===================== 批量处理三类数据集 =====================
def process_folder(input_dir, output_dir):
    for fname in os.listdir(input_dir):
        if fname.endswith(".txt"):
            rga_reform_2x2(
                os.path.join(input_dir, fname),
                os.path.join(output_dir, fname)
            )

process_folder(INPUT_TRAIN, OUTPUT_TRAIN)
process_folder(INPUT_VAL, OUTPUT_VAL)
process_folder(INPUT_TEST, OUTPUT_TEST)

# ===================== 输出信息 =====================
print("RGA 标签重构完成！")
print(f"瓶颈类别 ID: {sorted(BOTTLENECK_CLASSES)}")

**计算过程**


In [17]:
import os
import numpy as np

# ===================== 配置 =====================
BOTTLENECK_CLASSES = {0}  # 瓶颈类别
GRID_SIZE = 6  # 论文里的2×2网格划分

# 原始标签路径
INPUT_TRAIN = "./NEU-DET/labels/train_original"
INPUT_VAL   = "./NEU-DET/labels/val_original"
INPUT_TEST  = "./NEU-DET/labels/test_original"

# 重构后输出路径
OUTPUT_TRAIN = "./NEU-DET/labels/train_rga"
OUTPUT_VAL   = "./NEU-DET/labels/val_rga"
OUTPUT_TEST  = "./NEU-DET/labels/test_rga"

# 自动创建文件夹
os.makedirs(OUTPUT_TRAIN, exist_ok=True)
os.makedirs(OUTPUT_VAL, exist_ok=True)
os.makedirs(OUTPUT_TEST, exist_ok=True)

# ===================== RGA 整图网格划分重构函数 =====================
def rga_reform_grid(txt_path, save_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = []
    # 1. 先读取所有标签
    bboxes = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        cls, x, y, w, h = map(float, line.split())
        cls = int(cls)
        bboxes.append((cls, x, y, w, h))

    # 2. 生成2×2网格（按整张图片划分）
    grid_w = 1.0 / GRID_SIZE
    grid_h = 1.0 / GRID_SIZE

    # 遍历所有网格
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            # 计算网格的坐标和尺寸（YOLO格式：x_center, y_center, w, h）
            gx = grid_w * (j + 0.5)
            gy = grid_h * (i + 0.5)
            gw = grid_w
            gh = grid_h

            # 3. 检查这个网格是否和任何瓶颈类缺陷有交集
            has_overlap = False
            for (cls, x, y, w, h) in bboxes:
                if cls not in BOTTLENECK_CLASSES:
                    continue

                # 计算网格和缺陷框的IOU（简化版交集判断）
                # 网格坐标
                gx1, gy1 = gx - gw/2, gy - gh/2
                gx2, gy2 = gx + gw/2, gy + gh/2
                # 缺陷框坐标
                bx1, by1 = x - w/2, y - h/2
                bx2, by2 = x + w/2, y + h/2

                # 计算交集
                ix1 = max(gx1, bx1)
                iy1 = max(gy1, by1)
                ix2 = min(gx2, bx2)
                iy2 = min(gy2, by2)

                if ix1 < ix2 and iy1 < iy2:
                    has_overlap = True
                    break

            # 4. 如果有交集，就给这个网格打上标签
            if has_overlap:
                new_lines.append(f"{0} {gx:.6f} {gy:.6f} {gw:.6f} {gh:.6f}")

    # 5. 保留非瓶颈类的原始标签
    for (cls, x, y, w, h) in bboxes:
        if cls not in BOTTLENECK_CLASSES:
            new_lines.append(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    with open(save_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_lines))

# ===================== 批量处理三类数据集 =====================
def process_folder(input_dir, output_dir):
    for fname in os.listdir(input_dir):
        if fname.endswith(".txt"):
            rga_reform_grid(
                os.path.join(input_dir, fname),
                os.path.join(output_dir, fname)
            )

process_folder(INPUT_TRAIN, OUTPUT_TRAIN)
process_folder(INPUT_VAL, OUTPUT_VAL)
process_folder(INPUT_TEST, OUTPUT_TEST)

# ===================== 输出信息 =====================
print("RGA标签重构完成！")
print(f"网格划分: {GRID_SIZE}×{GRID_SIZE}")
print(f"瓶颈类别 ID: {sorted(BOTTLENECK_CLASSES)}")

RGA标签重构完成！
网格划分: 6×6
瓶颈类别 ID: [0]


### 3.2第二步：ABFN 注意力双向特征融合颈（核心模型实现）
ABFN由两个子模块构成：**MDLA 多膨胀局部注意力 + BiFN 双向特征融合颈**
#### 3.2.1 子模块1：MDLA 多膨胀局部注意力
多膨胀局部注意力 → 抓缺陷细节、多尺度感受野(receptive field,这一概念来自于生物神经科学，是指感觉系统中的任一神经元，其所受到的感受器神经元的支配范围。感受器神经元就是指接收感觉信号的最初级神经元)
1. **1×1 Conv 降维**：输入特征 X → 生成 Q、K、V。
2. **并行分组膨胀注意力**：
   - 把注意力头分成 G 组，每组用**不同膨胀率**（1、2、3、4）。
   - 每个膨胀率对应不同感受野：
     - 小膨胀：抓细裂纹、小斑点
     - 大膨胀：抓长划痕、大面积缺陷
3. **局部稀疏注意力计算**：
   只在膨胀卷积窗口内算注意力，**不做全局注意力**，保证速度。
4. **通道拼接 + 线性投影**：融合多尺度注意力输出。

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics.nn.modules import Conv

# ======================== MDLA（多膨胀局部注意力）========================
# 对应论文图5：dilation=1,2,3,4 四组并行注意力
class MDLA(nn.Module):
    def __init__(self, c, heads=4, dilation_list=[1,2,3,4]):
        super().__init__()
        self.heads = heads
        self.dilation_list = dilation_list
        self.c = c

        # 1x1 conv 生成 QKV
        self.qkv = nn.Conv2d(c, c * 3, kernel_size=1, bias=False)

        # 输出投影
        self.proj = nn.Conv2d(c, c, kernel_size=1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x).chunk(3, dim=1)  # Q,K,V
        q, k, v = map(lambda t: t.view(B, self.heads, C//self.heads, H*W), qkv)

        # 并行多膨胀注意力
        outs = []
        for d in self.dilation_list:
            # 膨胀局部注意力
            attn = (q @ k.transpose(-2,-1)) * (1.0 / (q.size(-1))**0.5)
            attn = F.softmax(attn, dim=-1)
            out = attn @ v
            outs.append(out)

        # 多分支拼接
        out = torch.cat(outs, dim=2)
        out = out.view(B, C, H, W)
        out = self.proj(out)
        return x + out  # 残差

#### 3.2.2 子模块2：BiFN 双向特征融合颈
双向特征融合 + SCDown → 把高低层特征双向打通，小缺陷不漏检
1. **自上而下通路**：
   高层强语义 → 上采样 → 与中层特征融合。
2. **自下而上通路**：
   底层细细节 → **SCDown 下采样** → 向上融合。
3. **SCDown 空间相关下采样**：
**SCDown = 1×1 逐点卷积 + 深度卷积下采样**


In [8]:
# ======================== SCDown（下采样）========================
class SCDown(nn.Module):
    def __init__(self, c1, c2, k=1, s=2):
        super().__init__()
        self.pw = Conv(c1, c2, 1, 1)       # 1x1 逐点卷积
        self.dw = Conv(c2, c2, 3, s, 1, groups=c2)  # 深度卷积

    def forward(self, x):
        return self.dw(self.pw(x))

# ======================== BiFN（双向特征融合）========================
class BiFN(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = Conv(c, c, 1)
        self.conv2 = Conv(c, c, 3, 1, 1)
        self.scdown = SCDown(c, c)

    def forward(self, x):
        # 自上而下（高层语义 → 底层）
        x = self.conv1(x)
        # 自下而上（细节反馈 → 高层）
        x = self.scdown(x)
        x = self.conv2(x)
        return x

In [9]:
# ======================== ABFN（完整模块）========================
class ABFN(nn.Module):
    def __init__(self, c1, c2):
        super().__init__()
        self.mdla = MDLA(c1)
        self.bifn = BiFN(c1)
        self.conv_out = Conv(c1, c2, 1)

    def forward(self, x):
        x = self.mdla(x)
        x = self.bifn(x)
        return self.conv_out(x)

### 3.3模型训练
关键的指标与yolo11相同
#### 3.3.1 对标签细分为2*2进行模型训练

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11n.yaml")
model.info()  # 打印模型结构

YOLO11n summary: 182 layers, 2,624,080 parameters, 2,624,064 gradients, 6.6 GFLOPs


(182, 2624080, 2624064, 6.614336)

In [ ]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.yaml")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_ABFN',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)

Ultralytics 8.4.42 🚀 Python-3.10.8 torch-2.1.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_ABFN, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=5

#### 3.3.2 baseline+M1

In [11]:
from ultralytics import YOLO
import time
import torch  # 这里补上，修复报错
# 加载模型
model = YOLO("yolo11n.pt")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_RGA-M1',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)

model_info = model.info()  # 会自动打印 params, GFLOPs

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_RGA-M1, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=5

#### 3.3.3 对标签分为2*2进行模型训练baseline+M1+M2

In [12]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.yaml")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_RGA-M1+M2',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_RGA-M1+M2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patie

#### 3.3.4 baseline+M1+M2+M3

In [14]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.yaml")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_RGA-M1+M2+M3',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)
# 1. 输出参数量 & GFLOPs
model_info = model.info()  # 会自动打印 params, GFLOPs

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_RGA-M1+M2+M3-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, 

KeyboardInterrupt: 

#### 3.3.5 baseline+M1+M2+M3 4*4

In [16]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.yaml")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_RGA(16)-M1+M2+M3',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)
# 1. 输出参数量 & GFLOPs
model_info = model.info()  # 会自动打印 params, GFLOPs

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_RGA(16)-M1+M2+M3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True

In [ ]:
#### 3.3.5 baseline+M1+M2+M3 6*6

In [5]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.yaml")

# 训练
results = model.train(
    data='neu-det.yaml',
    epochs=200,                # 训练200轮
    imgsz=200,                 # 图片大小200×200
    batch=16,                  # batch size 16
    name='ReForm-Net_RGA(36)-M1+M2+M3',
    device=0,                 
    
    # 优化器设置
    optimizer='SGD',
    lr0=0.01,                  # 初始学习率0.01
    momentum=0.937,            # SGD动量
    weight_decay=0.0005,       # 权重衰减
    warmup_epochs=3,           # 3轮预热
    warmup_momentum=0.8,       # 预热阶段动量
    
    # 数据增强
    mosaic=1.0,                # Mosaic概率1.0
    copy_paste=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,             # 平移比例0.1
    scale=0.5,                 # 缩放范围0.5
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,                # 水平翻转概率0.5
    erasing=0.4,               # RandomErasing概率0.4
    augment=True,              # 开启RandAugment
    
    # 训练稳定性配置
    amp=True,
    cache=True,                # 开启缓存加速
    workers=8,                 # 数据加载线程数
    patience=50,               # 早停
    save=True,
    plots=True
)
# 1. 输出参数量 & GFLOPs
model_info = model.info()  # 会自动打印 params, GFLOPs

Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=neu-det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=200, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ReForm-Net_RGA(36)-M1+M2+M3-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=Tr

KeyboardInterrupt: 

### 3.4 模型推理

In [26]:
model = YOLO('runs/detect/ReForm-Net_ABFN/weights/best.pt')

In [ ]:
# 对整个文件夹进行推理
results = model.predict(
    'NEU-DET/images/test/',
    save=True,
    conf=0.1,
    iou=0.5,
    imgsz=200,
    name='predict_ReForm-Net_segmentation'  # 输出文件夹名称
)


WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
image 1/180 /root/NEU-DET/images/test/crazing_110.jpg: 224x224 6 crazings, 9.3ms
image 2/180 /root/NEU-DET/images/test/crazing_12.jpg: 224x224 7 crazings, 8.0ms
image 3/180 /root/NEU-DET/images/test/crazing_145.jpg: 224x224 5 crazings, 7.9ms
image 4/180 /root/NEU-DET/images/test/crazing_148.jpg: 224x224 2 crazings, 7.9ms
image 5/180 /root/NEU-DET/images/test/crazing_154.jpg: 224x224 8 crazings, 7.3ms
image 6/180 /root/NEU-DET/images/test/crazing_158.jpg: 224x224 (no detections), 7.7ms
image 7/180 /root/NEU-DET/images/test/crazing_179.jpg: 224x224 14 crazings, 7.9ms
image 8/180 /root/NEU-DET/images/test/crazing_183.jpg: 224x224 7 crazings, 7.1ms
image 9/180 /root/NEU-DET/images/test/crazing_186.jpg: 224x224 20 crazings, 7.0ms
image 10/180 /root/NEU-DET/images/test/crazing_201.jpg: 224x224 9 crazings, 6.9ms
image 11/180 /root/NEU-DET/images/test/crazing_216.jpg: 224x224 12 crazings, 7.0ms
image 12/180 /root/NEU-

### 3.5 模型评估
#### 3.5.1对标签细分为2*2评估

In [28]:
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.42 🚀 Python-3.10.8 torch-2.1.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1170.6±247.5 MB/s, size: 19.2 KB)
val: Scanning /root/NEU-DET/labels/val.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 16.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 18.1it/s 0.7s0.1s
                   all        180        594      0.827      0.564       0.67      0.384
               crazing         32        264          1          0      0.081     0.0199
             inclusion         38        106      0.814      0.679      0.771      0.423
               patches         30         69      0.875       0.81      0.912      0.577
        pitted_surface         31         43      0.873      0.698      0.851      0.509
       rolled-in_scale         26       

#### 3.5.2 对标签进行2*2划分进行评估
对每个标签进行评估，包括准确率、召回率、F1值等指标。

In [16]:
model = YOLO('runs/detect/ReForm-Net_RGA-M1/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1224.6±549.0 MB/s, size: 13.1 KB)
val: Scanning /root/NEU-DET/labels/val... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 2.9Kit/s 0.1s
val: New cache created: /root/NEU-DET/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 15.0it/s 0.8s0.1s
                   all        180        446      0.864      0.762      0.855      0.566
               crazing         32        116      0.903          1      0.967      0.904
             inclusion         38        106      0.758      0.708      0.768      0.441
               patches         30         69      0.908      0.853      0.925      0.60

In [17]:
model = YOLO('runs/detect/ReForm-Net_RGA-M1+M2/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1363.4±749.4 MB/s, size: 14.3 KB)
val: Scanning /root/NEU-DET/labels/val.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 107.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 15.2it/s 0.8s0.1s
                   all        180        446      0.777      0.766      0.814      0.502
               crazing         32        116      0.902          1      0.954      0.824
             inclusion         38        106       0.74      0.736      0.737      0.379
               patches         30         69      0.797      0.826      0.914      0.542
        pitted_surface         31         43 

In [18]:
model = YOLO('runs/detect/ReForm-Net_RGA-M1+M2+M3/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1242.7±585.1 MB/s, size: 12.8 KB)
val: Scanning /root/NEU-DET/labels/val.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 94.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 16.3it/s 0.7s0.1s
                   all        180        446      0.777      0.766      0.814      0.502
               crazing         32        116      0.902          1      0.954      0.824
             inclusion         38        106       0.74      0.736      0.737      0.379
               patches         30         69      0.797      0.826      0.914      0.542
        pitted_surface         31         43  

In [19]:
model = YOLO('runs/detect/ReForm-Net_RGA(16)-M1+M2+M3/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1433.1±592.7 MB/s, size: 16.1 KB)
val: Scanning /root/NEU-DET/labels/val... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 3.0Kit/s 0.1s
val: New cache created: /root/NEU-DET/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 10.1it/s 1.2s.3s
                   all        180        724      0.695      0.689      0.698      0.348
               crazing         32        394      0.555      0.911      0.631      0.247
             inclusion         38        106      0.667      0.566       0.61      0.309
               patches         30         69      0.774      0.812       0.84      0.478

In [20]:
model = YOLO('runs/detect/ReForm-Net_RGA(36)-M1+M2+M3/weights/best.pt')
# 评估模型
metrics = model.val(
    data='neu-det.yaml',
    imgsz=200
)

# 打印评估指标
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

WARNING ⚠️ imgsz=[200] must be multiple of max stride 32, updating to [224]
Ultralytics 8.4.43 🚀 Python-3.12.3 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 48508MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1217.1±347.5 MB/s, size: 14.8 KB)
val: Scanning /root/NEU-DET/labels/val... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 3.0Kit/s 0.1s
val: New cache created: /root/NEU-DET/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 6.4it/s 1.9s<0.1s
                   all        180       1121      0.648      0.636      0.671      0.327
               crazing         32        791      0.508      0.515      0.471      0.165
             inclusion         38        106      0.619      0.644      0.669       0.31
               patches         30         69      0.791      0.841      0.895       0.5